# TravelTide: RFM and LRFM Input Preparation
Recent Frequency and Monetary are combined with historical booking Recency and relationship Length. Identical eligible customers and identical shared scaling parameters support a controlled RFM/LRFM comparison. All cohort members remain in the profiling table. No clustering is performed in this notebook.

Sources: data/raw/sessions.csv and the corrected files in data/own_processed. Outputs are separate from the original feature table. Candidate spend definitions retain the documented price assumptions and unknown-duration treatment.


In [ ]:
from pathlib import Path
from datetime import datetime
import math
import pandas as pd

# Run from the project root or its notebooks folder.
PROJECT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("Open this notebook from the traveltide_submission project.")
RAW = PROJECT / "data" / "raw"
OUT = PROJECT / "data" / "own_processed"
REPORT = PROJECT / "rfm_lrfm_preparation_summary.txt"
inputs = {
    "customers": OUT / "user_features_own.csv",
    "flights": OUT / "flights_own_processed.csv",
    "hotels": OUT / "hotels_own_processed.csv",
    "sessions": RAW / "sessions.csv",
}
for path in inputs.values():
    if not path.is_file():
        raise FileNotFoundError(f"Missing input: {path}. No outputs written.")
print("STEP 1: Reading existing study data...", flush=True)
d = pd.read_csv(inputs["customers"], dtype={"user_id":"string"}, low_memory=False)
f = pd.read_csv(inputs["flights"], dtype={"user_id":"string", "trip_id":"string"}, low_memory=False)
h = pd.read_csv(inputs["hotels"], dtype={"user_id":"string", "trip_id":"string"}, low_memory=False)
s = pd.read_csv(inputs["sessions"], dtype={"user_id":"string", "trip_id":"string", "session_id":"string"}, low_memory=False)
if d["user_id"].isna().any() or not d["user_id"].is_unique:
    raise ValueError("Customer IDs are not complete and unique; no outputs written.")

def boolean(series):
    parsed = series.astype("string").str.strip().str.lower().map(
        {"true":True,"false":False,"1":True,"0":False,"1.0":True,"0.0":False})
    if parsed.isna().any():
        raise ValueError(f"Missing or unrecognized boolean: {series.name}.")
    return parsed.astype(bool)

def date(series):
    return pd.to_datetime(series, format="mixed", errors="raise")

for c in ["cancellation","flight_booked","hotel_booked"]:
    s[c] = boolean(s[c])
s["session_start"] = date(s["session_start"])
if s["session_start"].isna().any():
    raise ValueError("Missing session-start date; no outputs written.")
if not s["session_id"].is_unique or not s["user_id"].isin(d["user_id"]).all():
    raise ValueError("Session IDs or customer mappings need review.")
START = pd.Timestamp("2023-01-05")
REFERENCE = s["session_start"].max().normalize()
END = REFERENCE + pd.Timedelta(days=1)
if not date(d["analysis_reference_date"]).eq(REFERENCE).all():
    raise ValueError("Reference dates differ: rebuild the customer features from this source snapshot first.")
canceled = set(s.loc[s["cancellation"],"trip_id"].dropna())
b = s.loc[~s["cancellation"] & s["trip_id"].notna()
          & (s["flight_booked"] | s["hotel_booked"])
          & ~s["trip_id"].isin(canceled)].copy()
if not b["trip_id"].is_unique:
    raise ValueError("Multiple surviving booking events per trip; no outputs written.")
for product, df, flag_col in [("flights",f,"flight_booked"),("hotels",h,"hotel_booked")]:
    if not df["trip_id"].is_unique or df["trip_id"].isna().any():
        raise ValueError(f"Duplicate or missing trip IDs in {product}.")
    expected = b.loc[b[flag_col], ["trip_id","user_id"]]
    if set(df["trip_id"]) != set(expected["trip_id"]):
        raise ValueError(f"Processed {product} and raw session bookings differ. Rebuild features first.")
    check = df[["trip_id","user_id"]].merge(expected,on="trip_id",validate="one_to_one",suffixes=("_product","_session"))
    if not check["user_id_product"].eq(check["user_id_session"]).all():
        raise ValueError(f"Customer mapping mismatch in {product}.")
print("STEP 1 OK", flush=True)
print("STEP 2: Calculating recent Frequency and Monetary...", flush=True)
recent = b.loc[b["session_start"].ge(START) & b["session_start"].lt(END)]
r = d.set_index("user_id").copy()
r["rfm_frequency_trips"] = recent.groupby("user_id")["trip_id"].nunique().reindex(r.index,fill_value=0)
# Historical noncanceled booking history supplies Recency and Length at the cutoff.
first = b.groupby("user_id")["session_start"].min().dt.normalize()
last = b.groupby("user_id")["session_start"].max().dt.normalize()
r["rfm_recency_days"] = (REFERENCE-last).dt.days
r["lrfm_length_days"] = (last-first).dt.days
unknown_total = pd.Series(0,index=r.index,dtype="int64")
for product,df,spend_col in [("flight",f,"flight_spend_estimate"),("hotel",h,"hotel_spend_estimate")]:
    part = df.loc[df["trip_id"].isin(recent["trip_id"])].copy()
    part[spend_col] = pd.to_numeric(part[spend_col],errors="coerce")
    if part[spend_col].dropna().lt(0).any():
        raise ValueError(f"Negative {product} expenditure; no outputs written.")
    counts = part.groupby("user_id")["trip_id"].nunique().reindex(r.index,fill_value=0)
    unknown = part[spend_col].isna().groupby(part["user_id"]).sum().reindex(r.index,fill_value=0).astype(int)
    subtotal = part.groupby("user_id")[spend_col].sum(min_count=1).reindex(r.index)
    subtotal.loc[counts.eq(0)] = 0.0
    r[f"recent_{product}_trip_count"] = counts
    r[f"recent_{product}_spend_unknown_booking_count"] = unknown
    r[f"recent_{product}_spend_estimate"] = subtotal.mask(unknown.gt(0))
    unknown_total = unknown_total + unknown
r["rfm_monetary_estimate"] = r["recent_flight_spend_estimate"]+r["recent_hotel_spend_estimate"]
r["recent_spend_complete"] = r["rfm_monetary_estimate"].notna()
r["feature_window_start"] = START.date().isoformat()
r["feature_window_end"] = REFERENCE.date().isoformat()
signup = date(r["sign_up_date"]).dt.normalize()
exposure_start = signup.where(signup.ge(START),START)
r["recent_potential_exposure_days"] = (END-exposure_start).dt.days
if r["recent_potential_exposure_days"].le(0).any():
    raise ValueError("Signup outside the observation window.")
model_cols = ["rfm_recency_days","rfm_frequency_trips","rfm_monetary_estimate","lrfm_length_days"]
eligible = r["rfm_frequency_trips"].gt(0) & r[model_cols].notna().all(axis=1)
r["baseline_status"] = "Ready for RFM/LRFM comparison"
r.loc[r["rfm_frequency_trips"].gt(0) & ~eligible,"baseline_status"] = "Recent booking; spending unresolved"
r.loc[r["rfm_frequency_trips"].eq(0) & r["valid_trip_count"].gt(0),"baseline_status"] = "Historical booking; no recent valid booking"
r.loc[r["rfm_frequency_trips"].eq(0) & r["valid_trip_count"].eq(0),"baseline_status"] = r.loc[r["rfm_frequency_trips"].eq(0) & r["valid_trip_count"].eq(0),"booking_status"]
assert int(r["rfm_frequency_trips"].sum()) == recent["trip_id"].nunique()
assert r["rfm_monetary_estimate"].isna().eq(unknown_total.gt(0)).all()
print("STEP 2 OK", flush=True)
print("STEP 3: Preparing matching RFM and LRFM inputs...", flush=True)
model = r.loc[eligible,model_cols].copy()
if len(model)<2:
    raise ValueError("Not enough complete recent bookers for model preparation.")
# Log transforms reduce skew; population standardization gives each feature equal scale.
# Applying math.log1p avoids the optional native correlation operation used in an earlier check.
transformed = model.apply(lambda column: column.map(math.log1p))
parameters=[]
scaled=pd.DataFrame(index=model.index)
for col in model_cols:
    values=transformed[col]
    mu=float(values.mean())
    sigma=math.sqrt(float(((values-mu)**2).mean()))
    if not math.isfinite(mu) or not math.isfinite(sigma) or sigma<=0:
        raise ValueError(f"Feature {col} is nonfinite or constant; no outputs written.")
    scaled[col]=(values-mu)/sigma
    parameters.append({"feature":col,"transform":"log1p","mean_after_transform":mu,"population_std_after_transform":sigma})
if not all(math.isfinite(float(v)) for v in scaled.to_numpy().ravel()):
    raise ValueError("Nonfinite modeling inputs; no outputs written.")
rfm_cols=model_cols[:3]
assert scaled[rfm_cols].index.equals(scaled.index)
print("STEP 3 OK", flush=True)
print("STEP 4: Saving separate outputs...", flush=True)
OUT.mkdir(parents=True,exist_ok=True)
r.reset_index().to_csv(OUT/"user_features_windowed.csv",index=False)
model.reset_index().to_csv(OUT/"rfm_lrfm_original_units.csv",index=False)
scaled[rfm_cols].reset_index().to_csv(OUT/"rfm_scaled_inputs.csv",index=False)
scaled.reset_index().to_csv(OUT/"lrfm_scaled_inputs.csv",index=False)
pd.DataFrame(parameters).to_csv(OUT/"rfm_lrfm_scaling_parameters.csv",index=False)
summary=[
    "TRAVELTIDE — RFM/LRFM INPUT PREPARATION",
    f"Generated: {datetime.now().astimezone().isoformat(timespec='seconds')}",
    f"Customer cohort: {len(r)} exported customers retained",
    f"Frequency/Monetary booking window: {START.date()} through {REFERENCE.date()}, inclusive",
    f"Historical Recency/Length reference date: {REFERENCE.date()}",
    "R = days since latest observed noncanceled booking, including earlier history.",
    "F = distinct noncanceled trips booked in the recent window.",
    "M = estimated expenditure on those recent bookings; not realized revenue or profit.",
    "L = first-to-last observed noncanceled booking span across the full export.",
    "These are deliberately different feature windows: recent purchasing behavior plus historical relationship context.",
    "Fare is assumed to cover the whole flight booking; hotel price is assumed per room-night; validated calendar nights are used.",
    "Unknown spending is preserved. No raw input or existing feature-build output is overwritten.",
    "Reruns replace only the files produced by this preparation notebook and its report.",
    "", "COVERAGE",
    *[f"{label}: {int(count)}" for label,count in r["baseline_status"].value_counts().items()],
    f"Recent valid trips: {int(r['rfm_frequency_trips'].sum())}",
    f"Matching RFM and LRFM modeling rows: {len(model)}",
    "", "EXPOSURE CAUTION",
    "Customers signed up on different dates. The common calendar window does not create equal individual exposure.",
    "Signup tenure and potential exposure remain in the profiling table; counts should not automatically be interpreted as loyalty.",
    f"Ready customers signed up after window start: {int(signup.loc[eligible].gt(START).sum())}",
    "", "MODEL PREPARATION",
    "Only R, F, M (RFM), and the additional L (LRFM) are included in the two input matrices.",
    "All dimensions are transformed with log1p and standardized on the same complete recent-booker subset.",
    "Identifiers are provided for rejoining labels but MUST NOT be passed as model features.",
    "No missing-value imputation, clustering, cluster naming, perk assignment, or outlier deletion was performed.",
    "Customers outside the initial modeling subset remain in user_features_windowed.csv for separate final handling.",
    "", "INPUT RANGES IN ORIGINAL UNITS: minimum / median / maximum",
]
for col in model_cols:
    summary.append(f"{col}: {model[col].min():.3f} / {model[col].median():.3f} / {model[col].max():.3f}")
summary += ["", "OUTPUTS", "user_features_windowed.csv", "rfm_lrfm_original_units.csv", "rfm_scaled_inputs.csv", "lrfm_scaled_inputs.csv", "rfm_lrfm_scaling_parameters.csv"]
REPORT.write_text("\n".join(summary),encoding="utf-8")
print(f"Preparation report: {REPORT}")
print("STEP 4 OK. Raw files and the existing feature table are unchanged.")
